<a href="https://colab.research.google.com/github/moloco/docs/blob/anoop-collab/Copy_of_CTV_Throttlers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import os
import sys
import time
from datetime import datetime, timedelta

import matplotlib
import numpy as np
import pandas as pd

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
from matplotlib import cm, pylab
import matplotlib.pyplot as plt

import subprocess

from google.cloud import storage

project='focal-elf-631'
client = bigquery.Client(project)

In [ ]:

br_result_dict = \
    {'CONFIGURED_NO_BID': '0',
     'THROTTLED_EVENTFNT_DOWN': '1-1',
     'THROTTLED_HIGH_SPEND_AVOIDANCE': '1-2',
     'THROTTLED_HIGH_BID_AVOIDANCE': '1-3',
     'THROTTLED_BOOTUP': '2-01',
     'THROTTLED_EXPERIMENT_TAG': '2-02',
     'THROTTLED_ABUSIVE': '2-03',
     'THROTTLED_CONFIG_SERVER_REGION': '2-04',
     'THROTTLED_NOT_ALLOWED_COUNTRY': '2-05',
     'THROTTLED_EXCHANGE': '2-06',
     'THROTTLED_EXCHANGE_INDIRECT_SOURCE': '2-07',
     'THROTTLED_CONFIG_BLOCKLIST': '2-08',
     'THROTTLED_ANONYMOUS_IP': '2-09',
     'THROTTLED_UNSUPPORTED_SDK_VERSION': '2-10',
     'THROTTLED_VI_FOR_CTV': '2-11',
     'THROTTLED_VIDEO_TYPE_THROTTLER': '2-12',
     'THROTTLED_MOLOCO_SDK_EXCLUSIVE_BUNDLE': '2-13',
     'THROTTLED_REQUEST_QUEUE_FULL': '3-1',
     'REQUEST_QUEUE_INTERNAL_ERROR': '3-2',
     'THROTTLED_EXPIRED_IN_REQUEST_QUEUE': '3-3',
     'THROTTLED_OUTSTANDING_REQUESTS': '3-4',
     'FILTERED_APT_READ': '4',
     'FILTERED_UNPUBLISHED_APP': '5-1',
     'FILTERED_PUBLISHER': '5-2',
     'FILTERED_SUPPRESSED_HEADER_BIDDING_REQUEST': '5-3',
     'THROTTLED_BY_TAGID_THROTTLER': '5-4',
     'THROTTLED_BID_CAP': '6-1',
     'THROTTLED_SHORT_TERM_USER_PROFILE': '6-2',
     'THROTTLED_PROBABILISTICALLY': '6-3',
     'THROTTLER_PASSED': '7'
     }

result_category_dict = \
  {
    '0': 'Configured no bid',
    '1': 'Anomaly detected',
    '2': 'Without state',
    '3': 'By system status',
    '4': 'Apt read',
    '5': 'By app data',
    '6': 'By user data',
    '7': 'Throttler passed'
  }
br_result_dict_rev = {value:key for key,value in br_result_dict.items()}
br_result_sr = pd.Series(br_result_dict)

result_category_sr =pd.Series(result_category_dict)
br_result_df = pd.DataFrame({'subcategory': br_result_sr})
br_result_df['category'] = [br_result_df.subcategory.iloc[i].split('-')[0] for i in range(len(br_result_df))]
br_result_df['result_cat'] = [result_category_dict[br_result_df.category.iloc[i]] for i in range(len(br_result_df))]


br_result_exclusion = [
    'TARGET_BID_NO_ACTIVE_TEST_CASE',  # target_bid
    'FILTERED_INCOMPATIBLE_TARGET_INVENTORY',  # target_bid
    'TARGET_BID_INTERNAL_ERROR',  # target_bid
    'INVALID_BID_REQUEST',  # bidreq
    'EXCHANGE_ON_PROBATION',  # bidreq
    'FILTER_BY_BIDFNT',  # bidreq
    'INTERNAL_ERROR',  # bidreq
]

post_throttle_results_dict = {
    # timeout_1 here
    'NO_CAMPAIGN_AFTER_REQ_FILTER': '2',
    'UPT_READ_ERROR': '3',
    # timeout_2 here
    'NO_CAMPAIGN_AFTER_CTX_FILTER': '5',
    'NO_ADGROUP_AFTER_CTX_FILTER': '6',
    # timeout_3 here
    'CONTEXT_TIMED_OUT': '8',
    'NO_WINNER': '9',
    'THROTTLED_POSTPRICING_WIN_PRED': 'A',
    # timeout_4 here
    # rendezvous results here
    'BID_SNAPSHOT_DEADLINE_EXCEEDED': 'D',
    'BID': 'E'
}
post_throttle_results_df = pd.DataFrame(data={'category':pd.Series(post_throttle_results_dict)})
timeout_df = pd.DataFrame(index=['get_candidate_campaigns', 'evaluate_candidate_campaigns', 'pick_winners', 'write_btlogs'],
                          data=['1','4','7','B'], columns=['category'])

In [ ]:
def post_throttle_result_gp(dt='250401', gp="", cond=""):
  query_str = """
select
  case
  """
  for i in range(len(timeout_df)):
    query_str += f"    when bid_result='TIME_OUT' and json_value(raw_json,'$.section_released_timestamp.{timeout_df.index[i]}') is null then '{timeout_df['category'].iloc[i]}.time_out'\n"
  for i in range(len(post_throttle_results_df)):
    query_str += f"    when bid_result='{post_throttle_results_df.index[i]}' then '{post_throttle_results_df['category'].iloc[i]}.{post_throttle_results_df.index[i].lower()}'\n"
  query_str += f"    when bid_result like 'RENDEZVOUS%' then 'C.rendezvous'"
  query_str += f"""
    else bid_result
  end as post_throttle_result,
     case when experiment_v2.assigned_group_ids like '%11528%' then 'control'
      when experiment_v2.assigned_group_ids like '%11529%' then 'test_throttle'
      when experiment_v2.assigned_group_ids like '%11530%' then 'test_no_throttle'
    else 'other' end as exp_group,
  sum(1/rate) cnt
  from `prod.trace20{dt}`
  where bid_result not in ('{"', '".join(br_result_exclusion)}')
  and bid_result not in ('{"', '".join(br_result_df.index)}')
  and ( experiment_v2.assigned_group_ids like '%11529%' OR experiment_v2.assigned_group_ids like '%11530%' )
  {cond}
  group by 1,2
  order by 1,2
  """
  print(query_str)
  x = client.query(query_str).to_dataframe()
  x0 = x.pivot(index='post_throttle_result', columns='exp_group', values='cnt').fillna(0)
  x0 = x0.sort_values('post_throttle_result')
  for col in x0.columns:
    x0[f'{col}_input'] = x0[col].iloc[::-1].cumsum(0).iloc[::-1]
  x1 = x0[[col for col in x0.columns if col.endswith('_input')]]
  x1.columns=[col.replace('_input','') for col in x1.columns]
  return x0,x1

In [ ]:
def throttle_result_gp(dt='250401', gp="", cond="", detailed=False):
  query_str = """
  select
    case
  """
  for i in range(len(br_result_sr)):
    if detailed:
      query_str += f"    when bid_result='{br_result_df.index[i]}' then '{br_result_df['subcategory'].iloc[i]}.{br_result_df.index[i].lower()}'\n"
    else:
      query_str += f"    when bid_result='{br_result_df.index[i]}' then '{br_result_df['category'].iloc[i]}.{br_result_df['result_cat'].iloc[i]}'\n"

  query_str += f"""
      else '7. Throttler passed'
    end as throttle_result,
    case when experiment_v2.assigned_group_ids like '%11528%' then 'control'
      when experiment_v2.assigned_group_ids like '%11529%' then 'test_throttle'
      when experiment_v2.assigned_group_ids like '%11530%' then 'test_no_throttle'
    else 'other' end as exp_group,
    sum(1/rate) cnt
  from `prod.trace20{dt}`
  where bid_result not in ('{"', '".join(br_result_exclusion)}')
  {cond}
  and ( experiment_v2.assigned_group_ids like '%11529%' OR experiment_v2.assigned_group_ids like '%11530%' )
  group by 1,2
  order by 1,2
  """
  print(query_str)
  x = client.query(query_str).to_dataframe()
  x0 = x.pivot(index='throttle_result', columns='exp_group', values='cnt').fillna(0)
  x0 = x0.sort_values('throttle_result')
  for col in x0.columns:
    x0[f'{col}_input'] = x0[col].iloc[::-1].cumsum(0).iloc[::-1]
  x1 = x0[[col for col in x0.columns if col.endswith('_input')]]
  x1.columns=[col.replace('_input','') for col in x1.columns]
  return x0,x1

In [ ]:
import plotly.express as px
# (TODO: younghan) in case there are too many group element, show top x and remainder in the chart
def funnel(x, n=10):
  if x.empty:
    print("No data to plot.")
    return
  x_new = x.T.sort_values(x.index[0]).iloc[::-1].T
  if len(x_new.columns) > n:
    x_new['Other'] = x_new.iloc[:,n:].sum(1)
    x_cut = x_new[[col for col in x_new.columns[:n]]+['Other']]
  else:
    x_cut = x_new
  fig = px.funnel(x_cut, width=1600, height=1200)
  now=datetime.now().strftime('%y%m%d_%H%M%S')
  fig.update_layout(
    font=dict(
        size=30,  # Set the font size here
        color="Black"
    )
  )
  fig.show()

In [ ]:
# x0, x1 = throttle_result_gp('230510','exchange','',True)
# y0, y1 = throttle_result_gp('230629','inventory_format','and exchange="UNITY" and bid_region="US"',True)
y0, y1 = throttle_result_gp('250806','exchange','and lower(os) = "ctv" and exchange in ("MCTV", "FREE_WHEEL")',True)
funnel(y1)


  select
    case
      when bid_result='CONFIGURED_NO_BID' then '0.configured_no_bid'
    when bid_result='THROTTLED_EVENTFNT_DOWN' then '1-1.throttled_eventfnt_down'
    when bid_result='THROTTLED_HIGH_SPEND_AVOIDANCE' then '1-2.throttled_high_spend_avoidance'
    when bid_result='THROTTLED_HIGH_BID_AVOIDANCE' then '1-3.throttled_high_bid_avoidance'
    when bid_result='THROTTLED_BOOTUP' then '2-01.throttled_bootup'
    when bid_result='THROTTLED_EXPERIMENT_TAG' then '2-02.throttled_experiment_tag'
    when bid_result='THROTTLED_ABUSIVE' then '2-03.throttled_abusive'
    when bid_result='THROTTLED_CONFIG_SERVER_REGION' then '2-04.throttled_config_server_region'
    when bid_result='THROTTLED_NOT_ALLOWED_COUNTRY' then '2-05.throttled_not_allowed_country'
    when bid_result='THROTTLED_EXCHANGE' then '2-06.throttled_exchange'
    when bid_result='THROTTLED_EXCHANGE_INDIRECT_SOURCE' then '2-07.throttled_exchange_indirect_source'
    when bid_result='THROTTLED_CONFIG_BLOCKLIST' then '2-

In [ ]:
y1

,ADX,ADX_RTB,APS,FREE_WHEEL,INDEX,MAGNITE,MCTV,NEXXEN,PUBMATIC,VUNGLE
throttle_result,,,,,,,,,,
0.configured_no_bid,47600000.0,74700000.0,18600000.0,2.775150e+10,8.203700e+09,1000000.0,3.488940e+10,9.854600e+09,5600000.0,14300000.0
1-1.throttled_eventfnt_down,47600000.0,74700000.0,18600000.0,2.774990e+10,8.201700e+09,1000000.0,3.488800e+10,9.854200e+09,5600000.0,14300000.0
1-3.throttled_high_bid_avoidance,47600000.0,74600000.0,18500000.0,2.772270e+10,8.189200e+09,1000000.0,3.485700e+10,9.845700e+09,5500000.0,14300000.0
2-01.throttled_bootup,47600000.0,74600000.0,18500000.0,2.772270e+10,8.189000e+09,1000000.0,3.485700e+10,9.845700e+09,5500000.0,14300000.0
2-05.throttled_not_allowed_country,47600000.0,74200000.0,18500000.0,2.763680e+10,8.168200e+09,1000000.0,3.474940e+10,9.819900e+09,5500000.0,14300000.0
2-06.throttled_exchange,47600000.0,71900000.0,18400000.0,2.763680e+10,8.165500e+09,1000000.0,3.470670e+10,9.819900e+09,5200000.0,14300000.0
2-08.throttled_config_blocklist,47600000.0,71800000.0,18400000.0,1.944520e+10,8.165300e+09,1000000.0,2.467670e+10,3.176500e+09,5200000.0,14300000.0
2-09.throttled_anonymous_ip,47600000.0,71800000.0,18400000.0,1.932790e+10,8.145200e+09,1000000.0,2.460680e+10,3.172300e+09,5200000.0,14300000.0
2-11.throttled_vi_for_ctv,46600000.0,70400000.0,17400000.0,1.900390e+10,8.051400e+09,800000.0,2.413230e+10,3.045000e+09,4800000.0,13000000.0


In [ ]:
y0


gp,ADX,ADX_RTB,APS,FREE_WHEEL,INDEX,MAGNITE,MCTV,NEXXEN,PUBMATIC,VUNGLE,ADX_input,ADX_RTB_input,APS_input,FREE_WHEEL_input,INDEX_input,MAGNITE_input,MCTV_input,NEXXEN_input,PUBMATIC_input,VUNGLE_input
throttle_result,,,,,,,,,,,,,,,,,,,,
0.configured_no_bid,0.0,0.0,0.0,1.600000e+06,2.000000e+06,0.0,1.400000e+06,4.000000e+05,0.0,0.0,47600000.0,74700000.0,18600000.0,2.775150e+10,8.203700e+09,1000000.0,3.488940e+10,9.854600e+09,5600000.0,14300000.0
1-1.throttled_eventfnt_down,0.0,100000.0,100000.0,2.720000e+07,1.250000e+07,0.0,3.100000e+07,8.500000e+06,100000.0,0.0,47600000.0,74700000.0,18600000.0,2.774990e+10,8.201700e+09,1000000.0,3.488800e+10,9.854200e+09,5600000.0,14300000.0
1-3.throttled_high_bid_avoidance,0.0,0.0,0.0,0.000000e+00,2.000000e+05,0.0,0.000000e+00,0.000000e+00,0.0,0.0,47600000.0,74600000.0,18500000.0,2.772270e+10,8.189200e+09,1000000.0,3.485700e+10,9.845700e+09,5500000.0,14300000.0
2-01.throttled_bootup,0.0,400000.0,0.0,8.590000e+07,2.080000e+07,0.0,1.076000e+08,2.580000e+07,0.0,0.0,47600000.0,74600000.0,18500000.0,2.772270e+10,8.189000e+09,1000000.0,3.485700e+10,9.845700e+09,5500000.0,14300000.0
2-05.throttled_not_allowed_country,0.0,2300000.0,100000.0,0.000000e+00,2.700000e+06,0.0,4.270000e+07,0.000000e+00,300000.0,0.0,47600000.0,74200000.0,18500000.0,2.763680e+10,8.168200e+09,1000000.0,3.474940e+10,9.819900e+09,5500000.0,14300000.0
2-06.throttled_exchange,0.0,100000.0,0.0,8.191600e+09,2.000000e+05,0.0,1.003000e+10,6.643400e+09,0.0,0.0,47600000.0,71900000.0,18400000.0,2.763680e+10,8.165500e+09,1000000.0,3.470670e+10,9.819900e+09,5200000.0,14300000.0
2-08.throttled_config_blocklist,0.0,0.0,0.0,1.173000e+08,2.010000e+07,0.0,6.990000e+07,4.200000e+06,0.0,0.0,47600000.0,71800000.0,18400000.0,1.944520e+10,8.165300e+09,1000000.0,2.467670e+10,3.176500e+09,5200000.0,14300000.0
2-09.throttled_anonymous_ip,1000000.0,1400000.0,1000000.0,3.240000e+08,9.380000e+07,200000.0,4.745000e+08,1.273000e+08,400000.0,1300000.0,47600000.0,71800000.0,18400000.0,1.932790e+10,8.145200e+09,1000000.0,2.460680e+10,3.172300e+09,5200000.0,14300000.0
2-11.throttled_vi_for_ctv,200000.0,700000.0,17300000.0,0.000000e+00,0.000000e+00,600000.0,1.058000e+08,6.480000e+07,4500000.0,12600000.0,46600000.0,70400000.0,17400000.0,1.900390e+10,8.051400e+09,800000.0,2.413230e+10,3.045000e+09,4800000.0,13000000.0


In [ ]:
z0,z1 = throttle_result_gp('250806','inventory_format', 'and exchange="UNITY"', True)
funnel(z1)

<ipython-input-7-22f5f59f9cd4>:25: FutureWarning:

In a future version of pandas all arguments of DataFrame.pivot will be keyword-only.



In [ ]:
x4, x5 = post_throttle_result_gp('250806','exchange','and lower(os) = "ctv" and exchange in ("MCTV", "FREE_WHEEL")')
funnel(x5)


select
  case
      when bid_result='TIME_OUT' and json_value(raw_json,'$.section_released_timestamp.get_candidate_campaigns') is null then '1.time_out'
    when bid_result='TIME_OUT' and json_value(raw_json,'$.section_released_timestamp.evaluate_candidate_campaigns') is null then '4.time_out'
    when bid_result='TIME_OUT' and json_value(raw_json,'$.section_released_timestamp.pick_winners') is null then '7.time_out'
    when bid_result='TIME_OUT' and json_value(raw_json,'$.section_released_timestamp.write_btlogs') is null then 'B.time_out'
    when bid_result='NO_CAMPAIGN_AFTER_REQ_FILTER' then '2.no_campaign_after_req_filter'
    when bid_result='UPT_READ_ERROR' then '3.upt_read_error'
    when bid_result='NO_CAMPAIGN_AFTER_CTX_FILTER' then '5.no_campaign_after_ctx_filter'
    when bid_result='NO_ADGROUP_AFTER_CTX_FILTER' then '6.no_adgroup_after_ctx_filter'
    when bid_result='CONTEXT_TIMED_OUT' then '8.context_timed_out'
    when bid_result='NO_WINNER' then '9.no_winner'
    when

In [ ]:
x0[[col for col in x0.columns if not col.endswith('_input')]].sum(1), x2[[col for col in x0.columns if not col.endswith('_input')]].sum(1)

(throttle_result
 0.configured_no_bid                           4.241789e+07
 1-1.throttled_eventfnt_down                   4.000050e+05
 2-01.throttled_bootup                         9.998300e+07
 2-03.throttled_abusive                        3.595130e+09
 2-05.throttled_not_allowed_country            4.710865e+09
 2-06.throttled_exchange                       1.293283e+10
 2-07.throttled_exchange_indirect_source       1.425700e+09
 2-08.throttled_config_blocklist               7.733500e+08
 2-09.throttled_anonymous_ip                   1.968121e+10
 2-10.throttled_unsupported_sdk_version        1.515000e+08
 2-11.throttled_vi_for_ctv                     8.470000e+06
 2-12.throttled_video_type_throttler           1.579500e+10
 2-13.throttled_moloco_sdk_exclusive_bundle    9.375000e+07
 3-1.throttled_request_queue_full              8.240000e+06
 3-3.throttled_expired_in_request_queue        1.096972e+09
 3-4.throttled_outstanding_requests            1.101078e+10
 4.filtered_apt_read   